# 05 â€” Baselines
Regression: global mean + persistence (prev1 as next). Classification: majority class + persistence (prev1 flag). Same 5-fold CV protocol.

In [1]:

import sys, os
from pathlib import Path
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from src.config import SPLITS_DIR, RESULTS_DIR, TARGETS_REG, CV_FOLDS, SEED
from src.evaluate import reg_metrics

train = pd.read_csv(SPLITS_DIR / "train.csv")
kf = KFold(CV_FOLDS, shuffle=True, random_state=SEED)
reg_rows = []
for target in TARGETS_REG:
    y = train[target].reset_index(drop=True)
    mean_scores, pers_scores = [], []
    for tr, va in kf.split(y):
        mean_scores.append(reg_metrics(y.iloc[va], np.full(len(va), y.iloc[tr].mean())))
        pers_scores.append(reg_metrics(y.iloc[va], train[target.replace("next_", "prev1_")].iloc[va]))
    row = {"target": target}
    for n, sc in [("mean", mean_scores), ("persistence", pers_scores)]:
        for metric in ("mae", "rmse", "r2"):
            row[f"{metric}_mean_{n}"] = round(float(np.mean([s[metric] for s in sc])), 4)
    reg_rows.append(row)
reg_base = pd.DataFrame(reg_rows)
display(reg_base)
reg_base.to_csv(RESULTS_DIR / "baselines_reg_cv.csv", index=False)


,target,mae_mean_mean,rmse_mean_mean,r2_mean_mean,mae_mean_persistence,rmse_mean_persistence,r2_mean_persistence
0,next_cycle_length,2.7447,3.8407,-0.0008,1.3023,1.9870,0.7319
1,next_period_length,1.0302,1.2791,-0.0004,0.6332,0.8913,0.5142


## Classification baselines

In [3]:
from sklearn.model_selection import StratifiedKFold
from src.config import TARGET_CLF
from src.evaluate import clf_metrics

skf = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
y = train[TARGET_CLF].astype(int).reset_index(drop=True)
prev1 = train["prev1_irregular_flag"].astype(int).reset_index(drop=True)
maj, per = [], []
for tr, va in skf.split(train, y):
    maj.append(clf_metrics(y.iloc[va], np.zeros(len(va))))
    per.append(clf_metrics(y.iloc[va], prev1.iloc[va]))
clf_base = pd.DataFrame({
    "majority_class": {k: round(float(np.mean([m[k] for m in maj])), 4) for k in maj[0]},
    "persistence_prev1_flag": {k: round(float(np.mean([m[k] for m in per])), 4) for k in per[0]},
}).T.reset_index().rename(columns={"index": "model"}).assign(target=TARGET_CLF)
display(clf_base)
clf_base.to_csv(RESULTS_DIR / "baselines_clf_cv.csv", index=False)


,model,pr_auc,f1,roc_auc,target
0,majority_class,0.0909,0.0000,0.5000,next_is_irregular
1,persistence_prev1_flag,0.6160,0.7708,0.8708,next_is_irregular
